In [ ]:
!pip install optuna

In [ ]:
!mkdir -p /content/csv_data

In [ ]:
for year in range(2017, 2027):
    if year == 2026:
        month_range = range(1, 3)
    else:
        month_range = range(1, 13)

    for month in month_range:
        print(year, month)
        !wget -P /content/csv_data --content-disposition "https://climate.weather.gc.ca/climate_data/bulk_data_e.html?format=csv&stationID=52518&Year={year}&Month={month}&Day=14&timeframe=1&submit=Download+Data"

        # 48568

In [ ]:
import glob
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


folder_path = '/content/csv_data'

csv_files = glob.glob(os.path.join(folder_path, '*.csv'))

dataframes = [pd.read_csv(f) for f in csv_files]
merged_df = pd.concat(dataframes)

data = merged_df.sort_values(by=['Year', 'Month', 'Day'])

indecies_as_timeseries = pd.to_datetime(data['Date/Time (LST)'])
data = data.set_index(indecies_as_timeseries)

wind_direction_in_radians = np.radians(data['Wind Dir (10s deg)'] * 10)
data['Sine of Wind Direction'] = np.sin(wind_direction_in_radians)
data['Cosine of Wind Direction'] = np.cos(wind_direction_in_radians)

data['Wind Speed (m/s)'] = data['Wind Spd (km/h)'] / 3.6

month_in_radians = (data['Month'] * 2 * np.pi) / 12
data['Sine of Month'] = np.sin(month_in_radians)
data['Cosine of Month'] = np.cos(month_in_radians)

day_in_radians = (data['Day'] * 2 * np.pi) / 31
data['Sine of Day'] = np.sin(day_in_radians)
data['Cosine of Day'] = np.cos(day_in_radians)

hour_as_number = data['Time (LST)'].str[:2].astype(int)
time_in_radians = (hour_as_number * 2 * np.pi) / 23
data['Sine of Hour'] = np.sin(time_in_radians)
data['Cosine of Hour'] = np.cos(time_in_radians)

data = data.drop(columns=[
    'Wind Dir (10s deg)',
    'Wind Spd (km/h)',
    'Date/Time (LST)',
    'Month',
    'Day',
    'Time (LST)',
    'Longitude (x)',
    'Latitude (y)',
    'Station Name',
    'Climate ID',
    'Year',
    # 'Date/Time (LST)',
    'Flag',
    'Temp Flag',
    'Dew Point Temp Flag',
    'Rel Hum Flag',
    'Precip. Amount (mm)',
    'Precip. Amount Flag',
    'Wind Dir Flag',
    'Visibility Flag',
    'Stn Press Flag',
    'Hmdx',
    'Hmdx Flag',
    'Wind Chill',
    'Wind Chill Flag',
    'Wind Spd Flag',
    'Weather'
])



data = data.rename(columns={
    'Temp (°C)': 'Temperature (°C)',
    'Dew Point Temp (°C)': 'Dew Point Temperature (°C)',
    'Rel Hum (%)': 'Relative Humidity (%)',
    'Precip. Amount (mm)': 'Precipitation Amount (mm)',
    'Stn Press (kPa)': 'Station Pressure (kPa)'
})


dataset_features = data.drop(columns=['Wind Speed (m/s)'])
dataset_targets = data['Wind Speed (m/s)']
X_train, X_test, Y_train, Y_test = train_test_split(dataset_features, dataset_targets, test_size=0.15, shuffle=False, random_state=42) # could be 20

X_train, X_test, Y_train, Y_test = X_train.ffill(), X_test.ffill(), Y_train.ffill(), Y_test.ffill()

In [ ]:
import optuna

# Hyperparameters configuration

# MLR - none
def get_linear_regression_params(trial):
    return {}

# Lasso

def get_lasso_params(trial):
    return {
        "model__alpha": trial.suggest_categorical(
            "alpha",
            [10**i for i in range(-3, 4)]
        )
    }


# Ridge

def get_ridge_params(trial):
    return {
        "model__alpha": trial.suggest_categorical(
            "alpha",
            [10**i for i in range(-3, 4)]
        )
    }


# Elastic Net

def get_elastic_net_params(trial):
    return {
        "model__alpha": trial.suggest_categorical(
            "alpha",
            [10**i for i in range(-3, 4)]
        )
    }


# KNN

def get_knn_params(trial):
    return {
        "model__n_neighbors": trial.suggest_int("n_neighbours", 1, 11),
        "model__n_jobs": -1
    }


# Decision Tree

def get_decision_tree_params(trial):
    return {
        "model__max_depth": trial.suggest_int("max_depth", 1, 9)
    }


# Ada Boost

def get_ada_boost_params(trial):
    return {
        "model__learning_rate": trial.suggest_categorical(
            "learning_rate",
            [10**i for i in range(-5, 1)]
        )
    }

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


def create_lags(X, y, lags_per_feature: list[list[int]], windows_per_feature: list[list[int]], features: list[str]):
    X_working = X.copy()
    y_working = y.copy()

    if lags_per_feature:
        for (feature, lags) in zip(features, lags_per_feature):
            for lag in lags:
                if lag < 0:
                    raise ValueError("Lag values must be non-negative.")

                if feature == y_working.name:
                    X_working[f"{feature}: lag {lag}"] = y_working.shift(lag)
                else:
                    X_working[f"{feature}: lag {lag}"] = X_working[feature].shift(lag)

    if windows_per_feature:
        for (feature, windows) in zip(features, windows_per_feature):
            for window in windows:
                if window <= 0:
                    raise ValueError("Window values must be positive.")

                if feature == y_working.name:
                    rolled_series = y_working.shift(1).rolling(window=window)
                    X_working[f"{feature}: roll mean {window}"] = rolled_series.mean()
                else:
                    rolled_series = X_working[feature].shift(1).rolling(window=window)
                    X_working[f"{feature}: roll mean {window}"] = rolled_series.mean()


    X_processed = X_working.dropna()

    y_processed = y_working.loc[X_processed.index]

    return X_processed, y_processed

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer, QuantileTransformer, Normalizer, KBinsDiscretizer, PolynomialFeatures

from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from sklearn.pipeline import Pipeline

# outlier_removers = [HampelFilter(), None] - due to execution time
# feature_selectors = [SelectKBestDynamicK(), None]

scalers = [StandardScaler(), MinMaxScaler(), RobustScaler(), PowerTransformer(), QuantileTransformer(), None]
feature_engineers = [True, None]
time_lags = [
    {'simple': None,
     'rolling': None},
    {'simple': [[1, 2, 3, 4], [1, 2, 3, 4], [1, 2, 3, 4]],
     'rolling': None},
    {'simple': [[1, 2, 3, 4, 5, 14, 16, 24, 48, 72], [1, 2, 3, 4, 5, 14, 16, 24, 48, 72], [1, 2, 3, 4, 11, 16, 22, 24, 48, 72]],
     'rolling': None},
    {'simple': [[1, 2, 3, 4, 5, 14, 16, 24, 48, 72], [1, 2, 3, 4, 5, 14, 16, 24, 48, 72], [1, 2, 3, 4, 11, 16, 22, 24, 48, 72]],
     'rolling': [[4, 12, 24], [4, 12, 24], [5, 12, 24]]},
    {'simple': None,
     'rolling': [[4], [4], [5]]},
    {'simple': [[1, 2, 3, 4, 5, 14, 16], [1, 2, 3, 4, 5, 14, 16], [1, 2, 3, 4, 11, 16, 22]],
     'rolling': [[4, 12, 24], [4, 12, 24], [5, 12, 24]]}
]


feature_engineering_methods = {
    'Linear': PolynomialFeatures(degree=2),
    'Non-Linear': KBinsDiscretizer(n_bins=8, strategy='kmeans', encode='ordinal')
}


linear_models = [type(Lasso()), type(LinearRegression()), type(Ridge()), type(ElasticNet())]


models = [LinearRegression(), Lasso(), Ridge(), ElasticNet(), KNeighborsRegressor(), DecisionTreeRegressor(), AdaBoostRegressor()]
parameters = [get_linear_regression_params, get_lasso_params, get_ridge_params, get_elastic_net_params, get_knn_params, get_decision_tree_params, get_ada_boost_params]


preprocessing_variants = []

for scaler in scalers:
    for feature_engineering in feature_engineers:
        for time_lag in time_lags:
            for (model, params) in zip(models, parameters):

                feature_engineer = None

                if feature_engineering:
                    if type(model) in linear_models:
                        feature_engineer = feature_engineering_methods['Linear']
                    else:
                        feature_engineer = feature_engineering_methods['Non-Linear']

                pipeline_config = [
                    ('time_lags', time_lag),
                    ('feature_engineering', feature_engineer),
                    ('scaler', scaler),
                    ('model', model)
                ]

                preprocessing_variants.append([pipeline_config, params])

[[('time_lags', {'simple': None, 'rolling': None}), ('feature_engineering', KBinsDiscretizer(encode='ordinal', n_bins=8, strategy='kmeans')), ('scaler', StandardScaler()), ('normalizer', Normalizer()), ('model', GradientBoostingRegressor())], <function get_gradient_boosting_params at 0x799c29b84400>]


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


db_path = "sqlite:////content/drive/My Drive/<TARGET FOLDER>/optuna_data.db"

Mounted at /content/drive


In [ ]:
from sklearn.model_selection import cross_val_score
from optuna.pruners import NopPruner
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import train_test_split

import pandas as pd
import optuna


file_path = '/content/drive/MyDrive/<TARGET FOLDER>/results.csv'

time_series_cv_split = TimeSeriesSplit(n_splits=6)


def objective(trial, X, y, pipeline_config, parameters_function):

    params = parameters_function(trial)

    pipeline = Pipeline(pipeline_config)

    pipeline.set_params(**params)

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=time_series_cv_split,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    trial.set_user_attr("cv_scores", scores.tolist())

    rmse = -scores.mean() #could also use median

    return rmse


for (config, params_function) in preprocessing_variants:
    working_config = config.copy()

    working_X_train = X_train.copy()
    working_Y_train = Y_train.copy()
    working_X_test = X_test.copy()
    working_Y_test = Y_test.copy()

    trial_log_rows = [{
        "trial": config.copy(),
        "scores": 0
    }]

    print(trial_log_rows)

    study = optuna.create_study(
        direction='minimize',
        storage=db_path,
        load_if_exists=True,
        pruner=NopPruner()
    )


    if working_config[0]:
        simple_lags = working_config[0][1]['simple']
        rolling_lags = working_config[0][1]['rolling']

        features = ['Wind Speed (m/s)', 'Relative Humidity (%)', 'Station Pressure (kPa)']

        working_X_train, working_Y_train = create_lags(X_train, Y_train, lags_per_feature=simple_lags, windows_per_feature=rolling_lags, features=features)
        working_X_test, working_Y_test = create_lags(X_test, Y_test, lags_per_feature=simple_lags, windows_per_feature=rolling_lags, features=features)


    time_lags = working_config.pop(0)

    wrapped_objective = lambda trial: objective(trial, working_X_train, working_Y_train, working_config, params_function)

    study.optimize(
        wrapped_objective,
        n_trials=40
    )


    best_parameters = study.best_params

    for trial in study.trials:
        trial_cv_scores = trial.user_attrs["cv_scores"]

        trial_log_rows.append({
            "trial": trial.number,
            "scores": trial_cv_scores
        })

    trial_log_rows.append({
        "trial": best_parameters,
        "scores": 0
    })

    pd.DataFrame(trial_log_rows).to_csv(
        file_path,
        mode="a",
        header=False,
        index=False
    )

[I 2026-04-09 22:55:19,143] A new study created in RDB with name: no-name-6898359b-5315-400f-a644-20acfa11e8ee


[{'trial': [('time_lags', {'simple': None, 'rolling': None}), ('feature_engineering', KBinsDiscretizer(encode='ordinal', n_bins=8, strategy='kmeans')), ('scaler', StandardScaler()), ('normalizer', Normalizer()), ('model', GradientBoostingRegressor())], 'scores': 0}]


[I 2026-04-09 22:56:33,300] Trial 0 finished with value: 2.8382434097603935 and parameters: {'learning_rate': 0.001, 'max_depth': 5}. Best is trial 0 with value: 2.8382434097603935.
[I 2026-04-09 22:57:48,098] Trial 1 finished with value: 2.357737137534121 and parameters: {'learning_rate': 0.1, 'max_depth': 5}. Best is trial 1 with value: 2.357737137534121.
[I 2026-04-09 22:58:32,387] Trial 2 finished with value: 2.8586071757708464 and parameters: {'learning_rate': 0.001, 'max_depth': 3}. Best is trial 1 with value: 2.357737137534121.
[I 2026-04-09 22:59:03,120] Trial 3 finished with value: 2.868053062062201 and parameters: {'learning_rate': 0.001, 'max_depth': 2}. Best is trial 1 with value: 2.357737137534121.
[I 2026-04-09 22:59:47,369] Trial 4 finished with value: 2.624473988224844 and parameters: {'learning_rate': 0.01, 'max_depth': 3}. Best is trial 1 with value: 2.357737137534121.
[I 2026-04-09 23:01:37,707] Trial 5 finished with value: 2.8219710023465256 and parameters: {'learni

[{'trial': [('time_lags', {'simple': [[1, 2, 3, 4], [1, 2, 3, 4], [1, 2, 3, 4]], 'rolling': None}), ('feature_engineering', KBinsDiscretizer(encode='ordinal', n_bins=8, strategy='kmeans')), ('scaler', StandardScaler()), ('normalizer', Normalizer()), ('model', GradientBoostingRegressor(learning_rate=1e-05, max_depth=7))], 'scores': 0}]


[I 2026-04-09 23:45:05,717] Trial 0 finished with value: 1.7790271307366738 and parameters: {'learning_rate': 1, 'max_depth': 6}. Best is trial 0 with value: 1.7790271307366738.
[I 2026-04-09 23:48:33,302] Trial 1 finished with value: 2.9014102652703344 and parameters: {'learning_rate': 1e-05, 'max_depth': 8}. Best is trial 0 with value: 1.7790271307366738.
[I 2026-04-09 23:50:25,407] Trial 2 finished with value: 2.9014947606354955 and parameters: {'learning_rate': 1e-05, 'max_depth': 4}. Best is trial 0 with value: 1.7790271307366738.
[I 2026-04-09 23:52:19,185] Trial 3 finished with value: 1.3944253330864864 and parameters: {'learning_rate': 0.1, 'max_depth': 4}. Best is trial 3 with value: 1.3944253330864864.
[I 2026-04-09 23:52:50,034] Trial 4 finished with value: 2.765275024076605 and parameters: {'learning_rate': 0.001, 'max_depth': 1}. Best is trial 3 with value: 1.3944253330864864.
[I 2026-04-09 23:56:40,744] Trial 5 finished with value: 1.7019043548622397 and parameters: {'lea